In [53]:
import numpy as np
import pandas as pd
import os

def preprocess_13f_tables(data_dir='2013q2_form13f'):
    """
    Preprocess SUBMISSION.tsv and INFOTABLE.tsv, mapping target names to raw columns.
    
    Args:
        data_dir (str): Directory with TSV files.
        timestamp (str): Target timestamp (e.g., '2016-03-31').
    
    Returns:
        pd.DataFrame: Preprocessed dataset with specified columns.
    """

    # Files and their required columns (using raw names)
    files = {
        'SUBMISSION': {
            'file': 'SUBMISSION.tsv',
            'usecols': ['ACCESSION_NUMBER', 'FILING_DATE', 'SUBMISSIONTYPE', 'CIK', 'PERIODOFREPORT']
        },
        'INFOTABLE': {
            'file': 'INFOTABLE.tsv',
            'usecols': ['ACCESSION_NUMBER', 'NAMEOFISSUER', 'TITLEOFCLASS', 'CUSIP', 'FIGI', 'VALUE', 'SSHPRNAMT', 'SSHPRNAMTTYPE', 'PUTCALL']
        }
    }
    
    # Load and preprocess
    all_dfs = {}
    for name, info in files.items():
        file_path = os.path.join(data_dir, info['file'])
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"{info['file']} not found in {data_dir}.")
        
        print(f"Reading {info['file']} with selected columns...")
        df = pd.read_csv(file_path, sep='\t', usecols=info['usecols'],
                         dtype={'ACCESSION_NUMBER': str, 'CIK': str})
        
        all_dfs[name] = df
    
    # Preprocess SUBMISSION
    sub_df = all_dfs['SUBMISSION']
    
    # Filter CIKs with exactly one 13F-HR
    hr_counts = sub_df[sub_df['SUBMISSIONTYPE'] == '13F-HR'].groupby('CIK').size()
    hra_counts = sub_df[sub_df['SUBMISSIONTYPE'] == '13F-HR/A'].groupby('CIK').size()
    single_hr_ciks = hr_counts[hr_counts == 1].index
    print(f"\nCIKs with exactly one 13F-HR: {len(single_hr_ciks)}")
    print(f"CIKs excluded (multiple 13F-HR): {len(hr_counts[hr_counts > 1])}")
    
    #sub_df = sub_df[(sub_df['SUBMISSIONTYPE'] == '13F-HR') | (sub_df['SUBMISSIONTYPE'] == '13F-HR/A')]
    
    # Analyze multiple filings
    print(f"CIKs with multiple 13F-HR: {len(hr_counts[hr_counts > 1])}")
    print(f"CIKs with multiple 13F-HR/A: {len(hra_counts[hra_counts > 1])}", "\n")
    
    # Get SUBMISSION rows for single 13F-HR CIKs
    sub_df = sub_df[sub_df['CIK'].isin(single_hr_ciks)].copy()
    
    # For each CIK, select one 13F-HR or latest 13F-HR/A
    sub_df['FILING_DATE'] = pd.to_datetime(sub_df['FILING_DATE'])
    sub_df['PERIODOFREPORT'] = pd.to_datetime(sub_df['PERIODOFREPORT'])
    selected_filings = []
    for cik in single_hr_ciks:
        cik_filings = sub_df[sub_df['CIK'] == cik]
        hr_filing = cik_filings[cik_filings['SUBMISSIONTYPE'] == '13F-HR']
        hra_filings = cik_filings[cik_filings['SUBMISSIONTYPE'] == '13F-HR/A']
        
        if not hra_filings.empty:
            # Take latest 13F-HR/A
            latest_hra = hra_filings.loc[hra_filings['FILING_DATE'].idxmax()]
            selected_filings.append(latest_hra)
        else:
            # Take the single 13F-HR
            selected_filings.append(hr_filing.iloc[0])
            
    # Convert selected filings to DataFrame
    selected_sub_df = pd.DataFrame(selected_filings)
    
    selected_sub_df = selected_sub_df.drop_duplicates(subset=['ACCESSION_NUMBER'], keep='last')
    selected_sub_df = selected_sub_df.drop_duplicates(subset=['FILING_DATE', 'SUBMISSIONTYPE', 'CIK'], keep='last')
    selected_sub_df = selected_sub_df.dropna()
    selected_sub_df.info()
    #print('Length is: ', len(selected_sub_df['ACCESSION_NUMBER'].unique()))
    
    #print("Checking for duplicates in SUBMISSION...")
    #print('Duplicates: ', sub_df['CIK'].duplicated().sum())
    
    #grouped = sub_df[sub_df['CIK'].duplicated(keep=False)].groupby(['CIK'])
    #for group_name, group_data in grouped:
        #print(f"\nGroup: {group_name}")
        #print(group_data)
        #print("-" * 40)  # Separator between groups
    #sub_df = sub_df.drop_duplicates(subset=['ACCESSION_NUMBER', 'FILING_DATE', 'SUBMISSIONTYPE', 'CIK'])
    
    # Preprocess INFOTABLE
    info_df = all_dfs['INFOTABLE']
    
    # Inner merge with INFOTABLE
    merged_df = selected_sub_df.merge(info_df, on='ACCESSION_NUMBER', how='inner')
    merged_df = merged_df[(merged_df['SSHPRNAMTTYPE'] == 'SH') & (merged_df['PUTCALL'].isna()) &
                          (merged_df['VALUE'] > 0) & (merged_df['SSHPRNAMT'] > 0)]
    merged_df = merged_df.dropna(subset=['ACCESSION_NUMBER', 'CUSIP', 'VALUE', 'SSHPRNAMT', 'SSHPRNAMTTYPE'])
    merged_df = merged_df.drop_duplicates(subset=['ACCESSION_NUMBER', 'CUSIP', 'FIGI', 'VALUE', 'SSHPRNAMT', 'SSHPRNAMTTYPE'
                                                  , 'PUTCALL'], keep='last')
    print("\n")
    merged_df.info()
    #print('Length is: ', len(merged_df['ACCESSION_NUMBER'].unique()))
    #difference = np.setdiff1d(selected_sub_df['ACCESSION_NUMBER'].unique(), merged_df['ACCESSION_NUMBER'].unique())
    #print("Difference between two columns: ", difference)
    #print(info_df.head(100))
    
    # Resolve CIK conflicts
    if 'CIK_x' in merged_df.columns and 'CIK_y' in merged_df.columns:
        merged_df['CIK'] = merged_df['CIK_x'].where(merged_df['CIK_x'].notna(), merged_df['CIK_y'])
        merged_df = merged_df.drop(columns=['CIK_x', 'CIK_y'])
    
    # Ensure only target columns remain
    final_df = merged_df
    print("\n", final_df.head())
    
    # Check if file exists - if not, create it with headers
    try:
        tracking_df = pd.read_csv('trading_metrics.csv')
    except FileNotFoundError:
        tracking_df = pd.DataFrame(columns=['timestamp', 'unique_ciks', 'num_trades', 'unique_cusips'])

    # Get your metrics (replace these with your actual calculations)
    current_unique_ciks = len(merged_df['CIK'].unique())  # Number of unique CIKs
    current_num_trades = len(merged_df)                   # Number of trades (rows)
    current_unique_cusips = len(merged_df['CUSIP'].unique())         # Number of unique CUSIPs
    current_timestamp = data_dir

    # Create new row
    new_row = {
        'timestamp': current_timestamp,
        'unique_ciks': current_unique_ciks,
        'num_trades': current_num_trades,
        'unique_cusips': current_unique_cusips
    }

    # Append to dataframe
    tracking_df = tracking_df._append(new_row, ignore_index=True)

    # Save to CSV
    tracking_df.to_csv('trading_metrics.csv', index=False)

    print(f"\nAdded new entry to tracking file: {current_timestamp}, {current_unique_ciks} unique CIKs, {current_num_trades} trades, {current_unique_cusips} unique CUSIPs")
        
    return final_df

def main():
    data_dir = '2025q1_form13f'
    
    df = preprocess_13f_tables(data_dir)
    output_file = f'{data_dir}.csv'
    df.to_csv(output_file, index=False)
    print(f"\nPreprocessed dataset saved to {output_file}")

if __name__ == "__main__":
    main()

Reading SUBMISSION.tsv with selected columns...
Reading INFOTABLE.tsv with selected columns...


C:\Users\emadi\AppData\Local\Temp\ipykernel_3192\2704577508.py:37: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='\t', usecols=info['usecols'],



CIKs with exactly one 13F-HR: 7883
CIKs excluded (multiple 13F-HR): 82
CIKs with multiple 13F-HR: 82
CIKs with multiple 13F-HR/A: 33 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7883 entries, 8193 to 5930
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ACCESSION_NUMBER  7883 non-null   object        
 1   FILING_DATE       7883 non-null   datetime64[ns]
 2   SUBMISSIONTYPE    7883 non-null   object        
 3   CIK               7883 non-null   object        
 4   PERIODOFREPORT    7883 non-null   datetime64[ns]
dtypes: datetime64[ns](2), object(3)
memory usage: 369.5+ KB


<class 'pandas.core.frame.DataFrame'>
Int64Index: 2712795 entries, 0 to 2932510
Data columns (total 13 columns):
 #   Column            Dtype         
---  ------            -----         
 0   ACCESSION_NUMBER  object        
 1   FILING_DATE       datetime64[ns]
 2   SUBMISSIONTYPE    object        
 3   CIK   